In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

# 베르누이 나이브 베이즈(Bernoulli Naive Baues) 알고리즘

베르누이 나이브 베이즈는 분류 데이터의 특징이 `0` 또는 `1`로 표현되었을 때 데이터의 출현 여부에 따라서 0 또는 1로 구분되는 데이터에 사용한다.

이메일 제목과 레이블(스팸 메일 여부)을 활용해서 베르누이 나이브 베이즈 분류로 스팸 메일을 분류한다.

## 스팸 메일 분류에 사용할 데이터를 준비한다.

간단한 스팸 메일 분류를 위해 다음과 같이 이메일 제목과 레이블이 붙어있는 데이터를 사용한다.  
email title: 이메일 제목, spam: 스팸 메일 여부(True => 스팸 메일, False => 일반 메일)

In [2]:
email_list = [
    {'email title' : 'free game only today', 'spam' : True},
    {'email title' : 'cheapest flight deal', 'spam' : True},
    {'email title' : 'limited time offer only today only today', 'spam' : True},
    {'email title' : 'today meeting schedule', 'spam' : False},
    {'email title' : 'your flight schedule attached', 'spam' : False},
    {'email title' : 'your credit card statement', 'spam' : False}
]

test_email_list = [
    {'email title' : 'free flight offer', 'spam' : True},
    {'email title' : 'hey traveler free flight deal', 'spam' : True},
    {'email title' : 'limited free game offer', 'spam' : True},
    {'email title' : 'today flight schedule', 'spam' : False},
    {'email title' : 'your credit card attached', 'spam' : False},
    {'email title' : 'free credit card offer only today', 'spam' : False}
]

학습 데이터(email_list)를 데이터프레임으로 만든다.

In [3]:
df = pd.DataFrame(email_list)
df

,email title,spam
0,free game only today,True
1,cheapest flight deal,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


사이킷런의 베르누이 나이브 베이즈 분류기는 숫자 데이터(0 또는 1)만 다룰 수 있기 때문에 True를 1로 False를 0으로 치환한 label이라는 파생 변수를 추가한다.

In [4]:
encode = LabelEncoder()
df['label'] = encode.fit_transform(df.spam)
df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deal,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


학습 데이터를 피처와 레이블로 분리한다.

In [5]:
x = df['email title']
x

0                        free game only today
1                        cheapest flight deal
2    limited time offer only today only today
3                      today meeting schedule
4               your flight schedule attached
5                  your credit card statement
Name: email title, dtype: str

In [6]:
y_train = df.label
y_train

0    1
1    1
2    1
3    0
4    0
5    0
Name: label, dtype: int64

모든 이메일 제목을 공백을 구분자로 지정해서 연결한 하나의 문자열을 다시 공백을 경계로 나눠서 중복을 제거한 오름차순으로 정렬된 리스트로 변환한다.

In [7]:
string = ' '.join(x)
print(string)
string = set(string.split())
print(string)
print(len(string))
string = list(string)
print(string)
string.sort()
print(string)

free game only today cheapest flight deal limited time offer only today only today today meeting schedule your flight schedule attached your credit card statement
{'only', 'meeting', 'today', 'cheapest', 'flight', 'time', 'card', 'free', 'offer', 'your', 'statement', 'game', 'deal', 'attached', 'schedule', 'credit', 'limited'}
17
['only', 'meeting', 'today', 'cheapest', 'flight', 'time', 'card', 'free', 'offer', 'your', 'statement', 'game', 'deal', 'attached', 'schedule', 'credit', 'limited']
['attached', 'card', 'cheapest', 'credit', 'deal', 'flight', 'free', 'game', 'limited', 'meeting', 'offer', 'only', 'schedule', 'statement', 'time', 'today', 'your']


## 모든 이메일 제목에 출현한 단어 개수 만큼의 크기를 가지는 고정된 벡터를 만든다. => 인코딩

모든 데이터에 출현한 단어의 개수 만큼의 크기를 가지는 벡터를 만들고 고정된 벡터로 표현하기 위해 CountVectorizer를 import 한다.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

CountVectorizer 클래스로 넘겨준 문자열에 출현한 모든 단어를 중복되는 단어는 제거하고 오름차순으로 정렬해서 단어의 위치로 행렬을 만들고 특정 단어가 출현할 경우 출현한 단어의 개수를 리턴하고 출현하지 않으면 0을 리턴한다.

`binary`의 기본값은 False이고 출현한 단어의 개수를 리턴하고 True로 변경하면 같은 단어가 여러번 출현하더라도 무조건 1을 리턴한다.

In [9]:
# CountVectorizer 클래스 객체를 만든다.
cv = CountVectorizer(binary=True)
# CountVectorizer 클래스 객체를 학습시켜서 결과를 적용한다.
# cv.fit(x)
# x_train = cv.transform(x)
x_train = cv.fit_transform(x)
print(x_train) # 희소 행렬 형태가 출력된다.
# toarray() 메소드는 희소 행렬을 넘파이 배열로 변환한다.
encoding = x_train.toarray()
print(type(encoding))
print(encoding)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>
  Coords	Values
  (0, 6)	1
  (0, 7)	1
  (0, 11)	1
  (0, 15)	1
  (1, 2)	1
  (1, 5)	1
  (1, 4)	1
  (2, 11)	1
  (2, 15)	1
  (2, 8)	1
  (2, 14)	1
  (2, 10)	1
  (3, 15)	1
  (3, 9)	1
  (3, 12)	1
  (4, 5)	1
  (4, 12)	1
  (4, 16)	1
  (4, 0)	1
  (5, 16)	1
  (5, 3)	1
  (5, 1)	1
  (5, 13)	1
<class 'numpy.ndarray'>
[[0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0]
 [0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0]
 [1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1]
 [0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1]]


위의 넘파이 배열에서 볼 수 있듯이 이메일 제목에서 총 17개의 단어가 발견되어 각 이메일 제목이 17개 크기의 벡터로 인코딩(표현)된 것을 확인할 수 있다.  
binary=True를 지정해서 베르누이 나이브 베이즈의 입력으로 사용하기 위해 이메일 제목에 중복되는 단어가 있더라도 출현한 횟수로 표현하는 것이 아니고 1로 표현된 것을 알 수 있다.

CountVectorizer로 인코딩된 결과를 inverse_transform() 메소드로 디코딩한다.

In [10]:
for s in cv.inverse_transform(x_train):
    print(s)

['free' 'game' 'only' 'today']
['cheapest' 'flight' 'deal']
['only' 'today' 'limited' 'time' 'offer']
['today' 'meeting' 'schedule']
['flight' 'schedule' 'your' 'attached']
['your' 'credit' 'card' 'statement']


get_feature_names_out() 메소드로 고정된 벡터의 각 열이 어떤 단어들을 의미하는지 확인할 수 있다.

In [11]:
print(cv.get_feature_names_out())

['attached' 'card' 'cheapest' 'credit' 'deal' 'flight' 'free' 'game'
 'limited' 'meeting' 'offer' 'only' 'schedule' 'statement' 'time' 'today'
 'your']


## 베르누이 나이브 베이즈 모델을 만들고 학습시킨다.

사이킷런의 베르누이 나이브 베이즈 분류기는 기본적으로 라플라스 스무딩(laplace smoothing) 기능을 지원하므로 학습 데이터에 없는 단어가 테스트 데이터에 있어도 분류가 진행된다.

라플라스 스무딩은 0이라는 수는 곱셈과 나눗셈을 무력화시키는 값이므로 의미있는 값이 도출된다 하더라도 마지막에 0을 곱해버리면 값은 0이 나오게 된다. 이런 경우가 상당히 빈번하기 때문에 `0을 0이 아닌 최소값으로 보정`을 하는데 이를 라플라스 스무딩이라 한다.

베르누이 나이브 베이즈 알고리즘을 사용하기 위해 BernoulliNB를 import 한다.

In [12]:
from sklearn.naive_bayes import BernoulliNB

In [13]:
model = BernoulliNB()
model.fit(x_train, y_train)

BernoulliNB()

## 학습된 베르누이 나이브 베이즈 모델을 테스트 데이터로 예측한다.

테스트 데이터(test_email_list)를 데이터프레임으로 만들고 True를 1로 False를 0으로 치환한 label이라는 파생 변수를 추가한다.

In [14]:
df = pd.DataFrame(test_email_list)
df['label'] = encode.fit_transform(df.spam)
df

,email title,spam,label
0,free flight offer,True,1
1,hey traveler free flight deal,True,1
2,limited free game offer,True,1
3,today flight schedule,False,0
4,your credit card attached,False,0
5,free credit card offer only today,False,0


테스트 데이터를 피처와 레이블로 분리한다.

In [15]:
x = df['email title']
x

0                    free flight offer
1        hey traveler free flight deal
2              limited free game offer
3                today flight schedule
4            your credit card attached
5    free credit card offer only today
Name: email title, dtype: str

In [16]:
y_test = df.label
y_test

0    1
1    1
2    1
3    0
4    0
5    0
Name: label, dtype: int64

In [17]:
x_test = cv.transform(x)
print(x_test)
print(x_test.toarray())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>
  Coords	Values
  (0, 5)	1
  (0, 6)	1
  (0, 10)	1
  (1, 4)	1
  (1, 5)	1
  (1, 6)	1
  (2, 6)	1
  (2, 7)	1
  (2, 8)	1
  (2, 10)	1
  (3, 5)	1
  (3, 12)	1
  (3, 15)	1
  (4, 0)	1
  (4, 1)	1
  (4, 3)	1
  (4, 16)	1
  (5, 1)	1
  (5, 3)	1
  (5, 6)	1
  (5, 10)	1
  (5, 11)	1
  (5, 15)	1
[[0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0]
 [1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 1 0 1 0 0 1 0 0 0 1 1 0 0 0 1 0]]


In [18]:
predict = model.predict(x_test)
print(predict)

[1 1 1 0 0 1]


## 모델을 평가한다.

In [20]:
classification = classification_report(y_test, predict)
print(classification)

              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6

